#In this code file, we have evaluated and compared the performance of different models for downscaling using several performance metrics(RMSE, Correlation,PSNR, and MSSIM).

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import numpy as np
import glob
import cv2
import tensorflow as tf
import math
from skimage.metrics import structural_similarity as ssim
from matplotlib import pyplot as plt

In [11]:
path='/content/drive/My Drive/Work7_MAUNet_Light_Data_Model_Result/Downscaling'

In [12]:
GT_Label=np.load(path+'/Data/GTIndiaTestTrain.npy')

In [13]:
has_nan = np.isnan(GT_Label).any()
print(has_nan)

False


In [14]:
MAUNet_Light_KR_Test_Grid=np.load(path+'/Result/Result_MAUNet_Light_KR.npy')
MAUNet_Light_GT_Test_Grid=np.load(path+'/Result/Result_MAUNet_Light_GT.npy')
MAUNet_Light_MP_Test_Grid=np.load(path+'/Result/Result_MAUNet_Light_MP.npy')
Label_Test_Grid=GT_Label[-854:]

In [19]:
np.shape(Label_Test_Grid)

(854, 140, 140)

In [16]:
datamask = np.load(path+'/Data/Datamask140.npy')

In [23]:
import numpy.ma as ma
totalmask=[]
for i in range(len(Label_Test_Grid)):
  totalmask.append(datamask)

In [25]:

MAUNet_Light_KR_Test_Grid_Mask=ma.masked_array(MAUNet_Light_KR_Test_Grid,mask=totalmask)
MAUNet_Light_GT_Test_Grid_Mask=ma.masked_array(MAUNet_Light_GT_Test_Grid,mask=totalmask)
MAUNet_Light_MP_Test_Grid_Mask=ma.masked_array(MAUNet_Light_MP_Test_Grid,mask=totalmask)
Label_Test_Grid_Mask=ma.masked_array(Label_Test_Grid,mask=totalmask)


In [37]:
import numpy as np
import math
from sklearn.metrics import mean_squared_error

def calculate_rmse(GT_mask, result_mask):
    GT_mask = np.float32(GT_mask)
    result_mask = np.float32(result_mask)

    mse = []

    for i in range(len(GT_mask)):
        x = GT_mask[i]
        y = result_mask[i]

        # Calculate MSE only for unmasked values
        mse.append(mean_squared_error(x[~x.mask], y[~y.mask]))

    MSError = np.sum(mse) / len(GT_mask)
    rmse = math.sqrt(MSError)

    print("Rounded RMSE:", round(rmse, 4))

    return rmse

In [38]:

import numpy as np
from skimage.metrics import peak_signal_noise_ratio

def calculate_psnr(GT_mask, result_mask, data_range):
    GT_mask = np.float32(GT_mask)
    result_mask = np.float32(result_mask)

    # Use only unmasked values
    GT_valid = GT_mask[~GT_mask.mask]
    result_valid = result_mask[~result_mask.mask]

    psnr = peak_signal_noise_ratio(
        GT_valid,
        result_valid,
        data_range=np.max(data_range)
    )

    print("Rounded PSNR:", round(psnr, 4))

    return psnr

In [39]:
import numpy as np
from skimage.metrics import structural_similarity

def calculate_ssim(GT_mask, result_mask, data_range):
    GT_mask = np.float32(GT_mask)
    result_mask = np.float32(result_mask)

    ssim = []

    for i in range(len(GT_mask)):
        x = GT_mask[i]
        y = result_mask[i]

        # Use only unmasked values
        x_valid = x[~x.mask]
        y_valid = y[~y.mask]

        sm = structural_similarity(
            x_valid,
            y_valid,
            data_range=np.max(data_range)
        )

        ssim.append(sm)

    mssim = np.sum(ssim) / len(GT_mask)

    print("Rounded Mean SSIM:", round(mssim, 4))

    return mssim

In [40]:
import numpy as np
import scipy.stats.mstats as cm

def calculate_pearson_correlation(GT_mask, result_mask):
    GT_mask = np.float32(GT_mask)
    result_mask = np.float32(result_mask)

    corr = cm.pearsonr(result_mask, GT_mask)

    print("Pearson correlation:", np.round(corr[0], 4))

    return corr[0]

In [41]:
#Performance Measures MAUNet_Light_KR
rmse_KR=calculate_rmse(Label_Test_Grid_Mask,MAUNet_Light_KR_Test_Grid_Mask)
psnr_KR=calculate_psnr(Label_Test_Grid_Mask,MAUNet_Light_KR_Test_Grid_Mask, np.max(GT_Label))
mssim_KR=calculate_ssim(Label_Test_Grid_Mask,MAUNet_Light_KR_Test_Grid_Mask, np.max(GT_Label))
corr_KR=calculate_pearson_correlation(Label_Test_Grid_Mask,MAUNet_Light_KR_Test_Grid_Mask)

Rounded RMSE: 11.4286
Rounded PSNR: 37.1341
Rounded Mean SSIM: 0.8725
Pearson correlation: 0.7199


In [42]:
#Performance Measures MAUNet_Light_MP
rmse_MP=calculate_rmse(Label_Test_Grid_Mask,MAUNet_Light_MP_Test_Grid_Mask)
psnr_MP=calculate_psnr(Label_Test_Grid_Mask,MAUNet_Light_MP_Test_Grid_Mask, np.max(GT_Label))
mssim_MP=calculate_ssim(Label_Test_Grid_Mask,MAUNet_Light_MP_Test_Grid_Mask, np.max(GT_Label))
corr_MP=calculate_pearson_correlation(Label_Test_Grid_Mask,MAUNet_Light_MP_Test_Grid_Mask)

Rounded RMSE: 11.4515
Rounded PSNR: 37.1167
Rounded Mean SSIM: 0.8717
Pearson correlation: 0.7196


In [43]:
#Performance Measures MAUNet_Light_GT
rmse_GT=calculate_rmse(Label_Test_Grid_Mask,MAUNet_Light_GT_Test_Grid_Mask)
psnr_GT=calculate_psnr(Label_Test_Grid_Mask,MAUNet_Light_GT_Test_Grid_Mask, np.max(GT_Label))
mssim_GT=calculate_ssim(Label_Test_Grid_Mask,MAUNet_Light_GT_Test_Grid_Mask, np.max(GT_Label))
corr_GT=calculate_pearson_correlation(Label_Test_Grid_Mask,MAUNet_Light_GT_Test_Grid_Mask)

Rounded RMSE: 11.5365
Rounded PSNR: 37.0524
Rounded Mean SSIM: 0.8665
Pearson correlation: 0.7107
